# Landslide step 02: direct damages (minimum_scenario)

Mirrors the coastal direct-damage workflow while applying landslide source-zone logic:
- Any landslide probability `>= 0.5` is treated as a source zone at risk
- Damage ratio is fixed at `1.0` for source-zone pixels


In [ ]:
import logging
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd


In [ ]:
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')

data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'
network_csv = data_root / 'networks/network_layers_hazard_intersections_details.csv'

shared_intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/landslide_hazard_network_intersections'
hazard_csv = shared_intersections_path / 'landslide_rasters_for_intersections.csv'

output_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_minimum_scenario'
output_path.mkdir(parents=True, exist_ok=True)
damage_results_folder = output_path / 'direct_damages'
damage_results_folder.mkdir(parents=True, exist_ok=True)

project_data_root = data_root
network_layers_input_file = project_data_root / 'networks/network_layers_hazard_intersections_details.csv'
network_layers_table = pd.read_csv(network_layers_input_file)
network_layers_table = network_layers_table[['path']].drop_duplicates().reset_index(drop=True)
network_layers_table['path'] = network_layers_table['path'].str.replace(r'^networks/', 'networks/networks/', regex=True)
network_layers_output_file = output_path / 'network_layers_for_intersections.csv'
network_layers_table.to_csv(network_layers_output_file, index=False)

hazard_layers_table = pd.read_csv(hazard_csv)
hazard_layers_output_file = output_path / 'landslide_rasters_for_intersections.csv'
hazard_layers_table.to_csv(hazard_layers_output_file, index=False)

vector_details_csv = network_layers_output_file
raster_details_csv = hazard_layers_output_file
hazard_csv = hazard_layers_output_file

sensitivity_csv = output_path / 'sensitivity_parameters.csv'
pd.DataFrame([
    {
        'cost_uncertainty_parameter': 0.0,
        'damage_uncertainty_parameter': 0.0,
    }
]).to_csv(sensitivity_csv, index=False)

print('Network layers file:', vector_details_csv)
print('Hazard layers file:', raster_details_csv)
print('Summary hazard file:', hazard_csv)
print('Using landslide minimum_scenario scenario: cost_uncertainty_parameter=0.0, damage_uncertainty_parameter=0.0')


In [ ]:
epsg_jamaica = 3448
jd_to_usd = 1 / 150  # exact: 150 JMD = 1 USD
source_zone_probability_threshold = 0.5
source_zone_damage_ratio = 1.0


def resolve_network_asset_file(asset_relative_path: str, data_root_path: Path) -> Path:
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root_path / relative_asset_path
    asset_file_in_nested_networks_folder = data_root_path / 'networks' / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: {asset_file_in_common_incoming_data} ; {asset_file_in_nested_networks_folder}"
    )


def convert_cost_units(row, cost_value_col: str, cost_unit_col: str, conversion_rate: float):
    unit_value = str(row[cost_unit_col])
    if ('US$' in unit_value) or ('USD' in unit_value):
        return conversion_rate * row[cost_value_col]
    return row[cost_value_col]


def modify_cost_units(row, cost_unit_col: str, damage_cost_col: str = 'damage_cost'):
    unit_value = str(row[cost_unit_col])
    if '/km' in unit_value:
        return 0.001 * row[damage_cost_col]
    return row[damage_cost_col]


def add_exposure_dimensions(gdf: gpd.GeoDataFrame, layer_type: str):
    if layer_type == 'edges':
        gdf['exposure'] = gdf.geometry.length
        gdf['exposure_unit'] = 'm'
    elif layer_type == 'areas':
        gdf['exposure'] = gdf.geometry.area
        gdf['exposure_unit'] = 'm2'
    else:
        gdf['exposure'] = 1.0
        gdf['exposure_unit'] = 'unit'

    gdf = gdf.drop(columns=['geometry'])
    index_columns = [column_name for column_name in gdf.columns if column_name != 'exposure']
    return gdf.groupby(index_columns, dropna=False)['exposure'].sum().reset_index()


def cleaned_damage_cost_unit(unit_series: pd.Series, layer_type: str) -> pd.Series:
    units = unit_series.fillna('J$').astype(str)
    if layer_type == 'nodes':
        return units
    return units.str.split('/').str[0]


asset_data_details = pd.read_csv(network_csv)
hazard_data_details = pd.read_csv(hazard_csv, encoding='latin1')
hazard_keys = hazard_data_details['key'].dropna().unique().tolist()
hazard_layers_name = Path(raster_details_csv).stem

sensitivity = pd.read_csv(sensitivity_csv)
cost_uncertainty_parameter = float(sensitivity.loc[0, 'cost_uncertainty_parameter'])
damage_uncertainty_parameter = float(sensitivity.loc[0, 'damage_uncertainty_parameter'])

processed_assets = []
missing_intersections = []

for asset_info in asset_data_details.itertuples(index=False):
    asset_id_col = asset_info.asset_id_column
    layer_type = asset_info.asset_layer

    intersection_file = shared_intersections_path / f"{asset_info.asset_gpkg}_splits__{hazard_layers_name}__{layer_type}.geoparquet"
    output_file = damage_results_folder / f"{asset_info.asset_gpkg}_{layer_type}" / f"{asset_info.asset_gpkg}_{layer_type}_direct_damages_parameter_set_0.parquet"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    if not intersection_file.exists():
        missing_intersections.append(str(intersection_file))
        continue

    asset_gpkg_file = resolve_network_asset_file(asset_info.path, data_root)
    asset_df = gpd.read_file(asset_gpkg_file, layer=layer_type)

    if asset_id_col not in asset_df.columns:
        raise KeyError(f"Asset ID column '{asset_id_col}' not found in {asset_gpkg_file} ({layer_type})")

    min_cost_col = asset_info.asset_min_cost_column if isinstance(asset_info.asset_min_cost_column, str) else None
    max_cost_col = asset_info.asset_max_cost_column if isinstance(asset_info.asset_max_cost_column, str) else None
    mean_cost_col = asset_info.asset_mean_cost_column if isinstance(asset_info.asset_mean_cost_column, str) else None
    cost_unit_col = asset_info.asset_cost_unit_column if isinstance(asset_info.asset_cost_unit_column, str) else None

    if not cost_unit_col or cost_unit_col not in asset_df.columns:
        asset_df['_tmp_cost_unit'] = 'J$'
        cost_unit_col = '_tmp_cost_unit'

    for maybe_cost_col in [min_cost_col, max_cost_col, mean_cost_col]:
        if maybe_cost_col and maybe_cost_col in asset_df.columns:
            asset_df[maybe_cost_col] = pd.to_numeric(asset_df[maybe_cost_col], errors='coerce').fillna(0.0)

    if min_cost_col and min_cost_col in asset_df.columns:
        asset_df[min_cost_col] = asset_df.apply(
            lambda row: convert_cost_units(row, min_cost_col, cost_unit_col, 1.0 / jd_to_usd),
            axis=1,
        )
    if max_cost_col and max_cost_col in asset_df.columns:
        asset_df[max_cost_col] = asset_df.apply(
            lambda row: convert_cost_units(row, max_cost_col, cost_unit_col, 1.0 / jd_to_usd),
            axis=1,
        )

    asset_df[cost_unit_col] = asset_df[cost_unit_col].replace(['USD', 'US$'], 'J$', regex=True)

    if asset_info.sector == 'energy' and layer_type == 'edges' and 'length' in asset_df.columns:
        if min_cost_col and min_cost_col in asset_df.columns:
            asset_df[min_cost_col] = np.where(asset_df['length'] > 0, asset_df[min_cost_col] / asset_df['length'], 0.0)
        if max_cost_col and max_cost_col in asset_df.columns:
            asset_df[max_cost_col] = np.where(asset_df['length'] > 0, asset_df[max_cost_col] / asset_df['length'], 0.0)
        asset_df[cost_unit_col] = 'J$/m'

    if min_cost_col and min_cost_col in asset_df.columns and max_cost_col and max_cost_col in asset_df.columns:
        asset_df['damage_cost'] = asset_df[min_cost_col] + cost_uncertainty_parameter * (asset_df[max_cost_col] - asset_df[min_cost_col])
    elif mean_cost_col and mean_cost_col in asset_df.columns:
        asset_df['damage_cost'] = asset_df[mean_cost_col]
    elif min_cost_col and min_cost_col in asset_df.columns:
        asset_df['damage_cost'] = asset_df[min_cost_col]
    elif max_cost_col and max_cost_col in asset_df.columns:
        asset_df['damage_cost'] = asset_df[max_cost_col]
    else:
        asset_df['damage_cost'] = 0.0

    asset_df['damage_cost'] = asset_df.apply(lambda row: modify_cost_units(row, cost_unit_col), axis=1)

    asset_lookup = asset_df[[asset_id_col, 'damage_cost', cost_unit_col]].copy()

    hazard_df = gpd.read_parquet(intersection_file)
    hazard_df = hazard_df.to_crs(epsg=epsg_jamaica)
    hazard_df = add_exposure_dimensions(hazard_df, layer_type)

    missing_hazard_keys = [key for key in hazard_keys if key not in hazard_df.columns]
    if missing_hazard_keys:
        for key in missing_hazard_keys:
            hazard_df[key] = np.nan

    keep_columns = [asset_id_col, 'exposure', 'exposure_unit'] + hazard_keys
    hazard_df = hazard_df[keep_columns]

    hazard_df = pd.merge(hazard_df, asset_lookup, how='left', on=asset_id_col)
    hazard_df['damage_cost'] = pd.to_numeric(hazard_df['damage_cost'], errors='coerce').fillna(0.0)
    hazard_df[cost_unit_col] = hazard_df[cost_unit_col].fillna('J$')

    base_damage = hazard_df['damage_cost'] * hazard_df['exposure']

    for hazard_key in hazard_keys:
        hazard_probability = pd.to_numeric(hazard_df[hazard_key], errors='coerce').fillna(-np.inf)
        hazard_df[hazard_key] = np.where(
            hazard_probability >= source_zone_probability_threshold,
            base_damage * source_zone_damage_ratio,
            0.0,
        )

    hazard_df['damage_cost_unit'] = cleaned_damage_cost_unit(hazard_df[cost_unit_col], layer_type)

    sum_dict = {hazard_key: 'sum' for hazard_key in hazard_keys}
    output_df = (
        hazard_df.groupby(
            [asset_id_col, 'exposure_unit', 'damage_cost_unit', 'exposure'],
            dropna=False,
        )
        .agg(sum_dict)
        .reset_index()
    )

    output_df['damage_uncertainty_parameter'] = damage_uncertainty_parameter
    output_df['cost_uncertainty_parameter'] = cost_uncertainty_parameter

    output_df.to_parquet(output_file, index=False)

    processed_assets.append({
        'asset_gpkg': asset_info.asset_gpkg,
        'asset_layer': layer_type,
        'output_file': str(output_file),
        'rows_written': len(output_df),
    })

processed_assets_df = pd.DataFrame(processed_assets)
processed_assets_df.to_csv(output_path / 'direct_damage_output_status.csv', index=False)

if missing_intersections:
    print('Missing intersections:', len(missing_intersections))
    for missing_path in missing_intersections:
        print('  -', missing_path)

print('Finished landslide direct damage calculations')
print('Assets processed:', len(processed_assets_df))
processed_assets_df


In [ ]:
# Completion check for expected direct-damage outputs
expected_outputs = []
asset_data_details = pd.read_csv(network_csv)
for asset_info in asset_data_details.itertuples(index=False):
    expected_file = damage_results_folder / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}" / f"{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet"
    expected_outputs.append({
        'asset_gpkg': asset_info.asset_gpkg,
        'asset_layer': asset_info.asset_layer,
        'output_file': str(expected_file),
        'exists': expected_file.exists(),
    })

expected_outputs_df = pd.DataFrame(expected_outputs)
print('Expected files:', len(expected_outputs_df))
print('Files present:', int(expected_outputs_df['exists'].sum()))
expected_outputs_df
